In [2]:
# ==========================================================
# 02_baseline_logisticRegression.py
# Corrected Baseline Logistic Regression
# KNN Imputation + Scaling + SHAP
# ==========================================================

import pandas as pd
import numpy as np
import os
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ----------------------------------------------------------
# 1. Load Clean Dataset
# ----------------------------------------------------------
file_path = "../dataset/clean_dengue_dataset.csv"
df = pd.read_csv(file_path)

# ----------------------------------------------------------
# 2. Separate Features and Target
# ----------------------------------------------------------
target_column = "dengue_label"

X = df.drop(columns=[target_column])
y = df[target_column]

feature_names = X.columns.tolist()

# ----------------------------------------------------------
# 3. Train-Test Split
# ----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ----------------------------------------------------------
# 4. KNN Imputation
# Logistic Regression cannot handle NaN values
# ----------------------------------------------------------
imputer = KNNImputer(n_neighbors=5)

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# ----------------------------------------------------------
# 5. Standard Scaling
# ----------------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# ----------------------------------------------------------
# 6. Train Logistic Regression Model
# ----------------------------------------------------------
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# ----------------------------------------------------------
# 7. Predictions
# ----------------------------------------------------------
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# ----------------------------------------------------------
# 8. Evaluation Metrics
# ----------------------------------------------------------
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("BASELINE LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {roc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ----------------------------------------------------------
# 9. Create Output Folders
# ----------------------------------------------------------
os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

/Users/anurag/miniconda3/envs/shap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASELINE LOGISTIC REGRESSION RESULTS
Accuracy : 0.9192
F1 Score : 0.9398
ROC AUC  : 0.8987

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.83      0.88        69
           1       0.91      0.97      0.94       129

    accuracy                           0.92       198
   macro avg       0.92      0.90      0.91       198
weighted avg       0.92      0.92      0.92       198



In [3]:
# ----------------------------------------------------------
# 10. Save Metrics
# ----------------------------------------------------------
metrics_df = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [acc],
    "F1_Score": [f1],
    "ROC_AUC": [roc]
})

metrics_df.to_csv(
    "outputs/baseline_logisticRegression_metrics.csv",
    index=False
)

In [4]:
# ----------------------------------------------------------
# 11. SHAP Explanation
# ----------------------------------------------------------
explainer = shap.LinearExplainer(model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

# ----------------------------------------------------------
# 12. Mean Absolute SHAP Importance
# ----------------------------------------------------------
importance = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print("\nBaseline SHAP Feature Importance:\n")
print(importance_df)

importance_df.to_csv(
    "outputs/baseline_logisticRegression_shap_importance.csv",
    index=False
)


Baseline SHAP Feature Importance:

                       Feature  Importance
3                    wbc_count    1.634532
6               platelet_count    0.599832
2              hemoglobin_g_dl    0.213529
0                          age    0.211588
7  platelet_distribution_width    0.107947
1                       gender    0.065782
5                    rbc_count    0.031889
4           differential_count    0.001265


In [ ]:
# ----------------------------------------------------------
# 13. SHAP Summary Plot
# ----------------------------------------------------------
import matplotlib.pyplot as plt

plt.figure()

shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=feature_names,
    show=True  # or just remove this argument
)

plt.tight_layout()
plt.show()   # explicitly display

# ----------------------------------------------------------
# 14. SHAP Force Plot (Instance 0)
# ----------------------------------------------------------
instance_index = 0

# Create force plot
shap.force_plot(
    explainer.expected_value,
    shap_values[instance_index],
    X_test_scaled[instance_index],
    feature_names=feature_names,
    matplotlib=True
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2749834925.py, line 28)